# 2. Exploración inicial de los datos

**Issue:** [#2 Exploración inicial de datos](https://github.com/velascocafe23/telco-churn-mlops/issues/2)

El objetivo de este notebook no es analizar el problema todavía, sino dejar el esquema
en condiciones: verificar qué es cada columna, unificar la representación de los valores
ausentes, asignar el tipo correcto a cada atributo y persistir el resultado en un formato
eficiente para las etapas siguientes.

El análisis exploratorio propiamente dicho se hace en el notebook 3, sobre el archivo
que produce este.

## 2.1 Configuración y carga

In [1]:
from pathlib import Path

import pandas as pd

RAW_FILE = Path("../../data/01_raw/telco_customer_churn.csv")
INTERMEDIATE_DIR = Path("../../data/02_intermediate")
INTERMEDIATE_FILE = INTERMEDIATE_DIR / "telco_customer_churn.parquet"

IDENTIFICADOR = "customerID"
OBJETIVO = "Churn"

COLUMNAS_NUMERICAS = ["tenure", "MonthlyCharges", "TotalCharges"]

COLUMNAS_CATEGORICAS = [
    "gender",
    "SeniorCitizen",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod",
    "Churn",
]

datos = pd.read_csv(RAW_FILE)
datos.shape

(7043, 21)

## 2.2 Descripción general

Primera revisión del esquema tal como llega desde la fuente.

In [2]:
datos.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [3]:
resumen = pd.DataFrame(
    {
        "tipo_origen": datos.dtypes.astype(str),
        "valores_unicos": datos.nunique(),
        "nulos": datos.isna().sum(),
        "pct_nulos": (datos.isna().mean() * 100).round(2),
    }
)
resumen

,tipo_origen,valores_unicos,nulos,pct_nulos
customerID,str,7043,0,0.0
gender,str,2,0,0.0
SeniorCitizen,int64,2,0,0.0
Partner,str,2,0,0.0
Dependents,str,2,0,0.0
tenure,int64,73,0,0.0
PhoneService,str,2,0,0.0
MultipleLines,str,3,0,0.0
InternetService,str,3,0,0.0
OnlineSecurity,str,3,0,0.0


El conteo de nulos da cero en todas las columnas, pero eso **no** significa que el
conjunto esté completo. Significa que los ausentes no están representados como nulos.
Lo confirma el tipo de `TotalCharges`: es un valor monetario acumulado que llegó como
texto, lo que solo ocurre cuando alguna celda contiene algo que no es un número.

## 2.3 Unificación de la representación de los valores nulos

Se buscan las formas alternativas en que puede estar codificada la ausencia de dato:
cadenas vacías, cadenas de solo espacios y los marcadores textuales habituales.

In [4]:
columnas_texto = datos.select_dtypes(include=["object", "string"]).columns

marcadores = ["", " ", "NA", "N/A", "na", "null", "NULL", "?", "-", "unknown"]

hallazgos = {}
for columna in columnas_texto:
    serie = datos[columna].astype("string").str.strip()
    coincidencias = serie.isin(marcadores).sum()
    if coincidencias > 0:
        hallazgos[columna] = int(coincidencias)

hallazgos

{'TotalCharges': 11}

In [5]:
datos_limpios = datos.copy()

for columna in columnas_texto:
    serie = datos_limpios[columna].astype("string").str.strip()
    datos_limpios[columna] = serie.mask(serie.isin(marcadores), pd.NA)

datos_limpios.isna().sum().loc[lambda s: s > 0]

TotalCharges    11
dtype: int64

Los ausentes quedan ahora representados de una sola forma. El `strip` aplicado de paso
elimina espacios sobrantes en los valores de las categóricas, que de no corregirse
generarían categorías duplicadas al hacer el *encoding*.

## 2.4 Conversión de tipos

Cada columna debe quedar con un tipo uniforme y semánticamente correcto:

- `TotalCharges` a numérico de punto flotante.
- `SeniorCitizen` llega como 0/1, pero es una condición del cliente, no una magnitud.
  Se traduce a las mismas etiquetas que usan las demás binarias para que el tratamiento
  sea homogéneo.
- El resto de atributos descriptivos a tipo categórico.
- El identificador se mantiene como texto.

In [6]:
datos_limpios["TotalCharges"] = pd.to_numeric(datos_limpios["TotalCharges"], errors="coerce")

datos_limpios["SeniorCitizen"] = datos_limpios["SeniorCitizen"].map({0: "No", 1: "Yes"})

for columna in COLUMNAS_CATEGORICAS:
    datos_limpios[columna] = datos_limpios[columna].astype("category")

datos_limpios[IDENTIFICADOR] = datos_limpios[IDENTIFICADOR].astype("string")

datos_limpios.dtypes

customerID            string
gender              category
SeniorCitizen       category
Partner             category
Dependents          category
tenure                 int64
PhoneService        category
MultipleLines       category
InternetService     category
OnlineSecurity      category
OnlineBackup        category
DeviceProtection    category
TechSupport         category
StreamingTV         category
StreamingMovies     category
Contract            category
PaperlessBilling    category
PaymentMethod       category
MonthlyCharges       float64
TotalCharges         Float64
Churn               category
dtype: object

In [7]:
for columna in COLUMNAS_CATEGORICAS:
    categorias = datos_limpios[columna].cat.categories.tolist()
    print(f"{columna:20s} {len(categorias):2d}  {categorias}")

gender                2  ['Female', 'Male']
SeniorCitizen         2  ['No', 'Yes']
Partner               2  ['No', 'Yes']
Dependents            2  ['No', 'Yes']
PhoneService          2  ['No', 'Yes']
MultipleLines         3  ['No', 'No phone service', 'Yes']
InternetService       3  ['DSL', 'Fiber optic', 'No']
OnlineSecurity        3  ['No', 'No internet service', 'Yes']
OnlineBackup          3  ['No', 'No internet service', 'Yes']
DeviceProtection      3  ['No', 'No internet service', 'Yes']
TechSupport           3  ['No', 'No internet service', 'Yes']
StreamingTV           3  ['No', 'No internet service', 'Yes']
StreamingMovies       3  ['No', 'No internet service', 'Yes']
Contract              3  ['Month-to-month', 'One year', 'Two year']
PaperlessBilling      2  ['No', 'Yes']
PaymentMethod         4  ['Bank transfer (automatic)', 'Credit card (automatic)', 'Electronic check', 'Mailed check']
Churn                 2  ['No', 'Yes']


Vale la pena registrar un patrón que aparece en el listado anterior: varias columnas de
servicios tienen tres categorías, donde la tercera (`No internet service` o
`No phone service`) es funcionalmente equivalente a `No`, pero codifica además que el
cliente no tiene contratado el servicio base.

Esa redundancia es información, no un error de calidad, así que **no se corrige aquí**.
La decisión de colapsarla o conservarla pertenece a la etapa de ingeniería de atributos,
y se toma con la evidencia del análisis exploratorio.

## 2.5 Duplicados

In [8]:
filas_duplicadas = datos_limpios.duplicated().sum()
atributos = datos_limpios.drop(columns=[IDENTIFICADOR])
filas_duplicadas_sin_id = atributos.duplicated().sum()
identificadores_duplicados = datos_limpios[IDENTIFICADOR].duplicated().sum()

print(f"Filas idénticas:                    {filas_duplicadas}")
print(f"Filas idénticas ignorando el ID:    {filas_duplicadas_sin_id}")
print(f"Identificadores repetidos:          {identificadores_duplicados}")

Filas idénticas:                    0
Filas idénticas ignorando el ID:    22
Identificadores repetidos:          0


In [9]:
columnas_duplicadas = [
    (izquierda, derecha)
    for indice, izquierda in enumerate(datos_limpios.columns)
    for derecha in datos_limpios.columns[indice + 1 :]
    if datos_limpios[izquierda].equals(datos_limpios[derecha])
]

print(f"Pares de columnas idénticas: {columnas_duplicadas}")

Pares de columnas idénticas: []


El resultado exige una lectura cuidadosa. No hay filas completamente identicas ni
identificadores repetidos, pero si aparecen registros con **todos los atributos iguales**
y distinto identificador. Conviene caracterizarlos antes de decidir.

In [10]:
mascara = atributos.duplicated(keep=False)
coincidentes = datos_limpios[mascara]

print(f"Registros involucrados: {len(coincidentes)}")
print(f"Perfiles distintos:     {atributos[mascara].drop_duplicates().shape[0]}")
print()
print(coincidentes[["tenure", "Contract", "InternetService", "PhoneService", OBJETIVO]])

Registros involucrados: 42
Perfiles distintos:     20

      tenure        Contract InternetService PhoneService Churn
22         1  Month-to-month              No          Yes   Yes
100        1  Month-to-month              No          Yes    No
542        1  Month-to-month              No          Yes    No
646        1  Month-to-month             DSL          Yes   Yes
662        1  Month-to-month              No          Yes    No
690        1  Month-to-month              No          Yes    No
964        1  Month-to-month             DSL          Yes   Yes
976        1  Month-to-month     Fiber optic          Yes   Yes
1243       1  Month-to-month             DSL          Yes   Yes
1338       1  Month-to-month              No          Yes   Yes
1491       1  Month-to-month              No          Yes    No
1731       1  Month-to-month     Fiber optic          Yes   Yes
1739       1  Month-to-month     Fiber optic          Yes   Yes
1932       1  Month-to-month              No     

## 2.6 Análisis de los valores ausentes detectados

La conversión de `TotalCharges` dejó al descubierto los registros que no eran numéricos.
Antes de decidir qué hacer con ellos hay que entender por qué están ausentes.

In [11]:
sin_total = datos_limpios[datos_limpios["TotalCharges"].isna()]

print(f"Registros sin TotalCharges: {len(sin_total)}")
sin_total[[IDENTIFICADOR, "tenure", "MonthlyCharges", "Contract", OBJETIVO]]

Registros sin TotalCharges: 11


,customerID,tenure,MonthlyCharges,Contract,Churn
488,4472-LVYGI,0,52.55,Two year,No
753,3115-CZMZD,0,20.25,Two year,No
936,5709-LVOEQ,0,80.85,Two year,No
1082,4367-NUYAO,0,25.75,Two year,No
1340,1371-DWPAZ,0,56.05,Two year,No
3331,7644-OMVMY,0,19.85,Two year,No
3826,3213-VVOLG,0,25.35,Two year,No
4380,2520-SGTTA,0,20.00,Two year,No
5218,2923-ARZLG,0,19.70,One year,No
6670,4075-WKNIU,0,73.35,Two year,No


In [12]:
print(f"Antigüedad de esos registros: {sorted(sin_total['tenure'].unique())}")
print(f"Clientes con tenure = 0 en total: {(datos_limpios['tenure'] == 0).sum()}")
print(f"Cancelaciones entre ellos: {(sin_total[OBJETIVO] == 'Yes').sum()}")

Antigüedad de esos registros: [np.int64(0)]
Clientes con tenure = 0 en total: 11
Cancelaciones entre ellos: 0


El resultado explica el ausente: todos los registros sin `TotalCharges` corresponden a
clientes con antigüedad cero, es decir, que aún no han recibido su primera factura. El
valor no falta por error de captura, **falta porque todavía no existe**.

Esto tiene dos consecuencias prácticas:

1. La imputación correcta es cero, no la media ni la mediana. Imputar con la mediana
   asignaría a un cliente recién ingresado un acumulado equivalente al de un cliente con
   dos años de historia, lo que introduce ruido y deteriora la señal de antigüedad.
2. La regla de validación que corresponde a esta columna no es "no permitir nulos", sino
   una regla de integridad entre campos: `TotalCharges` puede estar ausente únicamente
   cuando `tenure` es cero. Queda anotada para la etapa de *data validation*.

La imputación se implementa en el pipeline de ingeniería de atributos, no aquí, para que
quede dentro del objeto que se serializa junto al modelo y se aplique de forma idéntica
en entrenamiento y en inferencia.

## 2.7 Estadística descriptiva preliminar

In [13]:
datos_limpios[COLUMNAS_NUMERICAS].describe().round(2)

,tenure,MonthlyCharges,TotalCharges
count,7043.00,7043.00,7032.0
mean,32.37,64.76,2283.3
std,24.56,30.09,2266.77
min,0.00,18.25,18.8
25%,9.00,35.50,401.45
50%,29.00,70.35,1397.48
75%,55.00,89.85,3794.74
max,72.00,118.75,8684.8


In [14]:
datos_limpios[OBJETIVO].value_counts(normalize=True).round(4)

Churn
No     0.7346
Yes    0.2654
Name: proportion, dtype: float64

## 2.8 Persistencia del dataset intermedio

Se guarda en formato columnar Parquet, que conserva los tipos asignados. Volver a CSV
obligaria a repetir toda la conversion de tipos en cada notebook posterior, con el riesgo
de que la inferencia de tipos no sea consistente entre etapas.

La verificacion posterior contrasta el esquema escrito contra el releido. Se espera una
unica diferencia, en el identificador: pandas distingue entre el tipo `string` explicito
y el tipo `str` que asigna por defecto al leer, y el viaje por Parquet normaliza hacia el
segundo. Los valores no cambian. Lo que si debe conservarse estrictamente, y por eso se
verifica, es el tipo numerico de las tres magnitudes y el tipo categorico de los
diecisiete atributos descriptivos.


In [15]:
INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)
datos_limpios.to_parquet(INTERMEDIATE_FILE, index=False)

tamano_csv = RAW_FILE.stat().st_size / 1024
tamano_parquet = INTERMEDIATE_FILE.stat().st_size / 1024

print(f"Guardado en: {INTERMEDIATE_FILE.resolve()}")
print(f"CSV original: {tamano_csv:.1f} KB")
print(f"Parquet:      {tamano_parquet:.1f} KB")

Guardado en: /home/velas/proyectos/telco-churn-mlops/data/02_intermediate/telco_customer_churn.parquet
CSV original: 948.5 KB
Parquet:      192.9 KB


In [16]:
verificacion = pd.read_parquet(INTERMEDIATE_FILE)

comparacion = pd.DataFrame(
    {
        "antes": datos_limpios.dtypes.astype(str),
        "despues": verificacion.dtypes.astype(str),
    }
)
comparacion["coincide"] = comparacion["antes"] == comparacion["despues"]

assert verificacion.shape == datos_limpios.shape, "La forma no coincide tras la escritura"
assert comparacion.loc[COLUMNAS_NUMERICAS, "coincide"].all(), "Tipos numericos alterados"
assert (verificacion[COLUMNAS_CATEGORICAS].dtypes == "category").all(), "Categoricas alteradas"

comparacion

,antes,despues,coincide
customerID,string,string,True
gender,category,category,True
SeniorCitizen,category,category,True
Partner,category,category,True
Dependents,category,category,True
tenure,int64,int64,True
PhoneService,category,category,True
MultipleLines,category,category,True
InternetService,category,category,True
OnlineSecurity,category,category,True


## 2.9 Conclusiones

1. El conjunto no traia nulos declarados, pero si ausentes codificados como texto en
   blanco. La representacion quedo unificada.
2. `TotalCharges` venia como texto por esa causa y quedo convertida a numerico.
3. `SeniorCitizen` venia como entero binario y quedo homologada con el resto de
   atributos binarios.
4. No hay columnas duplicadas y el identificador de cliente es unico. Si existen
   registros con atributos identicos entre si: son clientes distintos cuyo perfil
   coincide, consecuencia esperable de la baja cardinalidad de las variables. **No se
   eliminan**, porque cada fila corresponde a un cliente real y descartarlos sesgaria la
   muestra hacia los perfiles menos frecuentes.
5. Los once ausentes de `TotalCharges` son estructurales: corresponden exactamente a los
   once clientes con antiguedad cero, que aun no han recibido factura. Ninguno de ellos
   registra cancelacion, lo que es coherente con su antiguedad. La imputacion con cero se
   hara en el pipeline de atributos.
6. Quedan registradas dos reglas para la etapa de validacion de datos: la integridad
   entre `TotalCharges` y `tenure`, y la unicidad del identificador de cliente.

**Siguiente paso:** analisis exploratorio sobre `data/02_intermediate/telco_customer_churn.parquet`.